# IFC Space Mapping — Data Preparation

This notebook prepares the **space mapping data** required by the Campus Digital Twin before any sensor data is fetched.

Run the cells **in order**. The output files produced here are consumed by later Python modules and Blender.

## What this notebook produces

| Step | Output file | Description |
|------|-------------|-------------|
| 1 | — | Inspect a single space to understand the IFC structure |
| 2 | `data/space_mapping.json` | All spaces with GUID, room number, name, and floor |
| 3 | `data/walls_with_spaces.json` | Walls with their adjacent space references |
| 4 | `data/walls_with_space_mapping.json` | Walls enriched with space mapping data |
| 5 | `data/space_related_elements.json` | Each space with all related building elements |

## Prerequisites

- `ifcopenshell` must be installed (see cell below).
- The IFC file (`ARK_MET_F1.ifc`) must be placed in the **same directory as this notebook**.
  > **⚠️ Note:** The IFC filename is currently hardcoded as `ARK_MET_F1.ifc`. Change it in each cell if your file has a different name.

## Install dependency

`ifcopenshell` requires the `lark` parser. Run this once per environment.

In [1]:
!pip install lark
!pip install numpy

---
## Step 1 — Inspect a single IfcSpace

**Purpose:** Understand what attributes and property sets a space exposes before extracting them in bulk.

Run this cell first to verify your IFC file loads correctly and to confirm which fields (`Name`, `LongName`, property sets) are populated.

> **TODO:** If the space at index `[1]` does not look representative, change the index to another space that matters for your mapping.

In [10]:
# Step 1: Inspect a single IfcSpace
# Use this to understand the data structure of your IFC file before bulk extraction.

import ifcopenshell
import ifcopenshell.util.element
import json

# ⚠️ Change this filename if your IFC model has a different name
IFC_FILE = 'ARK_MET_F1.ifc'


def inspect_space_parameters(ifc_file_path):
    model = ifcopenshell.open(ifc_file_path)
    spaces = model.by_type('IfcSpace')

    if not spaces:
        print("No spaces found in the file.")
        return

    # Inspect the second space (index 1). Change the index to explore other spaces.
    space = spaces[1]
    print(f"--- Inspecting Space: {space.Name} [{space.GlobalId}] ---\n")

    # 1. Direct Attributes
    # Fixed fields: GlobalId, Name, LongName, Description, ObjectType, etc.
    print("[1] CORE ATTRIBUTES:")
    attributes = space.get_info()
    for attr, value in attributes.items():
        print(f"    {attr}: {value}")

    # 2. Property Sets (Psets)
    # Where Revit parameters and any custom sensor assignments usually live.
    # If this section is empty, the IFC has no Psets on spaces — that is normal.
    print("\n[2] PROPERTY SETS (Parameters):")
    psets = ifcopenshell.util.element.get_psets(space)
    for pset_name, properties in psets.items():
        print(f"    Set: {pset_name}")
        for prop_name, prop_value in properties.items():
            print(f"        - {prop_name}: {prop_value}")

    # 3. Spatial context
    # Shows which storey or zone contains this space.
    print("\n[3] SPATIAL CONTEXT:")
    container = ifcopenshell.util.element.get_container(space)
    if container:
        print(f"    Contained in: {container.is_a()} - {container.Name}")


inspect_space_parameters(IFC_FILE)

--- Inspecting Space: C1504 [2yMDhecf15ff0Qukb9SbDr] ---

[1] CORE ATTRIBUTES:
    id: 530897
    type: IfcSpace
    GlobalId: 2yMDhecf15ff0Qukb9SbDr
    OwnerHistory: #6=IfcOwnerHistory(#3,#5,$,.NOCHANGE.,$,$,$,0)
    Name: C1504
    Description: None
    ObjectType: None
    ObjectPlacement: #529097=IfcLocalPlacement(#69,#529096)
    Representation: #530896=IfcProductDefinitionShape($,$,(#530895))
    LongName: Yleisöaula
    CompositionType: ELEMENT
    InteriorOrExteriorSpace: INTERNAL
    ElevationWithFlooring: None

[2] PROPERTY SETS (Parameters):

[3] SPATIAL CONTEXT:


---
## Step 2 — Extract all spaces → `data/space_mapping.json`

**Purpose:** Build the core space registry. Every subsequent step references spaces by their `space_guid`.

The script also reports **duplicate space numbers** (same `Name` on multiple spaces), which must be resolved manually before sensor mapping can work reliably.

> **TODO (if duplicates appear):** Inspect each duplicate in your BIM authoring tool and either merge the spaces or update the `floor_name` / `floor_guid` fields below to differentiate them.

**Fields written per space:**
- `space_guid` — IFC GlobalId (primary key used everywhere else)
- `space_number` — short room code (e.g. `C1504`), from `Name`
- `space_name` — descriptive name (e.g. `Yleisöaula`), from `LongName`
- `floor_guid` — currently fixed to `"01"` (update if the IFC contains multiple storeys)
- `floor_name` — currently fixed to `"Ground_Floor"` (update to match your IFC storey)

In [ ]:
# Step 2: Extract all IfcSpace entries and save to data/space_mapping.json
# This is the primary output of this notebook — used by all other scripts.

import ifcopenshell
import json
import os
from collections import defaultdict

# ⚠️ Change this filename if your IFC model has a different name
IFC_FILE = 'ARK_MET_F1.ifc'


def process_ifc_spaces(ifc_file_path):
    # Create output directory if needed
    data_dir = 'data'
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    json_file_path = os.path.join(data_dir, 'space_mapping.json')

    if not os.path.exists(ifc_file_path):
        print(f"Error: File {ifc_file_path} not found.")
        return

    model = ifcopenshell.open(ifc_file_path)
    spaces = model.by_type('IfcSpace')

    results = []
    for space in spaces:
        space_data = {
            "space_guid":   space.GlobalId,
            "space_number": space.Name,                      # Short room code used for sensor lookup
            "space_name":   getattr(space, 'LongName', None), # Human-readable name
            # TODO: Update floor_guid and floor_name if the IFC has multiple storeys.
            # You can derive these from ifcopenshell.util.element.get_container(space).
            "floor_guid":   "01",
            "floor_name":   "Ground_Floor"
        }
        results.append(space_data)

    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"--- Process Complete ---")
    print(f"File saved to: {json_file_path}")
    print(f"Total spaces found: {len(results)}")

    # Warn about duplicates — these need manual resolution before sensor mapping
    investigate_duplicates(results)


def investigate_duplicates(space_list):
    """Detect spaces that share the same space_number.
    Duplicates will cause ambiguous sensor-to-room matches in later steps.
    """
    number_groups = defaultdict(list)
    for space in space_list:
        num = space.get('space_number')
        number_groups[num].append(space)

    duplicates_found = False
    print(f"\n--- Investigating Duplicate Space Numbers ---")

    for num, instances in number_groups.items():
        if len(instances) > 1:
            duplicates_found = True
            print(f"Duplicate Found! Number: '{num}'")
            for inst in instances:
                print(f"  - GUID: {inst['space_guid']} | Name: {inst['space_name']}")
            print("-" * 20)

    if not duplicates_found:
        print("Success: No duplicate space numbers found.")


process_ifc_spaces(IFC_FILE)

--- Process Complete ---
File saved to: data\space_mapping.json
Total spaces found: 222

--- Investigating Duplicate Space Numbers ---
Duplicate Found! Number: 'A1502'
  - GUID: 3VTYFK8vD6zQrOpleLa9Rd | Name: Ruokailutilat
  - GUID: 2eKm1MRbj9lQ$fk6CjSQZn | Name: Ruokailutilat
--------------------
Duplicate Found! Number: '1'
  - GUID: 3dqv97I1b2JR60ciMz8rBL | Name: Huoneistoala 1-kerros
  - GUID: 02ZODw3PX4WPbIVq0FTGUh | Name: KOKOONTUMISTILA 1
  - GUID: 21VEzGVNLChPldQbstk9d4 | Name: LIIKETILA
--------------------
Duplicate Found! Number: '2'
  - GUID: 0QiU8cOrnAZ8CrW1VZg$UA | Name: KERROSALA 1-kerros
  - GUID: 02ZODw3PX4WPbIVq0FTGL3 | Name: KOKOONTUMISTILA 2
  - GUID: 21VEzGVNLChPldQbstk8H1 | Name: MAHDOLLINEN LIIKETILA
--------------------


---
## Step 3 — Extract walls with adjacent spaces → `data/walls_with_spaces.json`

**Purpose:** Discover which spaces border each wall. Two relationship types are used:
- `IfcRelSpaceBoundary` — explicit boundary definitions (most reliable)
- `IfcRelContainedInSpatialStructure` — fallback for walls directly contained in a storey

This file is an **intermediate step** — it is enriched in Step 4.

> **Note:** Walls with empty `spaces` arrays are contained in a storey but have no explicit space boundary. This is common for external walls.

In [1]:
# Step 3: Extract all walls and their adjacent spaces.
# Intermediate file — consumed by Step 4 to add space_mapping references.

import ifcopenshell
import json
import os
from collections import defaultdict

# ⚠️ Change this filename if your IFC model has a different name
IFC_FILE = 'ARK_MET_F1.ifc'


def extract_walls_with_spaces(ifc_path=IFC_FILE, out_path="data/walls_with_spaces.json"):
    model = ifcopenshell.open(ifc_path)

    # Collect both standard and legacy wall types
    walls = list(model.by_type("IfcWall")) + list(model.by_type("IfcWallStandardCase"))

    wall_to_spaces = defaultdict(list)

    # Primary source: explicit space boundary relationships
    for rel in model.by_type("IfcRelSpaceBoundary"):
        related = getattr(rel, "RelatedBuildingElement", None)
        space   = getattr(rel, "RelatingSpace", None)
        if not related or not space:
            continue
        if isinstance(related, (list, tuple)):
            for r in related:
                wall_to_spaces[r.GlobalId].append(space)
        else:
            wall_to_spaces[related.GlobalId].append(space)

    # Fallback: walls contained directly in a spatial structure (storey/zone)
    for rel in model.by_type("IfcRelContainedInSpatialStructure"):
        space = getattr(rel, "RelatingStructure", None)
        for el in getattr(rel, "RelatedElements", []) or []:
            if el.is_a("IfcWall") or el.is_a("IfcWallStandardCase"):
                wall_to_spaces[el.GlobalId].append(space)

    results = []
    for w in walls:
        spaces = wall_to_spaces.get(w.GlobalId, [])
        spaces_info = [
            {
                "global_id": s.GlobalId,
                "name":      getattr(s, "Name", None),
                "long_name": getattr(s, "LongName", None),
                "ifc_type":  s.is_a()
            }
            for s in spaces
        ]
        results.append({
            "wall_global_id": w.GlobalId,
            "wall_name":      getattr(w, "Name", None),
            "ifc_type":       w.is_a(),
            "spaces":         spaces_info
        })

    os.makedirs("data", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} walls -> {out_path}")
    return results


out = extract_walls_with_spaces()
print("Sample (first 3):")
print(json.dumps(out[:3], indent=2, ensure_ascii=False))

Saved 2649 walls -> data/walls_with_spaces.json
Sample (first 3):
[
  {
    "wall_global_id": "3w9amd8qrEXwHHbfc_$IZl",
    "wall_name": "Basic Wall:US1 (Ulkoseinä, teräsranka):481117",
    "ifc_type": "IfcWall",
    "spaces": [
      {
        "global_id": "0eoqdncsb8SAoZq$ETTO6Y",
        "name": "1.kerros",
        "long_name": "1.kerros",
        "ifc_type": "IfcBuildingStorey"
      }
    ]
  }
]


---
## Step 4 — Enrich walls with space_mapping → `data/walls_with_space_mapping.json`

**Purpose:** Cross-reference each wall's adjacent spaces against `space_mapping.json`.

Two match strategies are used in order:
1. **GUID match** — exact match on `space_guid` (most reliable)
2. **Name match** — fallback using normalised `space_number` or `space_name`

The `mapping_match_type` field in the output records which strategy succeeded (`"guid"`, `"space_number"`, or `null` if unmatched).

> **Prerequisite:** Run Step 2 first to produce `data/space_mapping.json`.

In [2]:
# Step 4: Link walls to space_mapping.json entries.
# Adds a 'mapping' field to each linked space so Blender can resolve
# wall -> space -> sensor in a single lookup.

import ifcopenshell
import json
import os
from collections import defaultdict

# ⚠️ Change this filename if your IFC model has a different name
IFC_FILE = 'ARK_MET_F1.ifc'


def link_walls_to_space_mapping(
    ifc_path=IFC_FILE,
    mapping_path="data/space_mapping.json",
    out_path="data/walls_with_space_mapping.json"
):
    if not os.path.exists(mapping_path):
        raise FileNotFoundError(
            f"{mapping_path} not found. Run Step 2 (space_mapping) first."
        )

    with open(mapping_path, "r", encoding="utf-8") as f:
        space_map = json.load(f)

    # Build two lookup tables for fast matching
    by_guid   = {entry.get("space_guid"): entry for entry in space_map if entry.get("space_guid")}
    def norm(x): return (x or "").strip().lower()
    by_number = {}
    for entry in space_map:
        num = norm(entry.get("space_number"))
        if num:
            by_number.setdefault(num, []).append(entry)

    model = ifcopenshell.open(ifc_path)
    walls = list(model.by_type("IfcWall")) + list(model.by_type("IfcWallStandardCase"))

    # Resolve wall -> adjacent spaces
    wall_to_spaces = defaultdict(list)
    for rel in model.by_type("IfcRelSpaceBoundary"):
        related = getattr(rel, "RelatedBuildingElement", None)
        space   = getattr(rel, "RelatingSpace", None)
        if not related or not space:
            continue
        if isinstance(related, (list, tuple)):
            for r in related:
                wall_to_spaces[r.GlobalId].append(space)
        else:
            wall_to_spaces[related.GlobalId].append(space)

    for rel in model.by_type("IfcRelContainedInSpatialStructure"):
        space = getattr(rel, "RelatingStructure", None)
        for el in getattr(rel, "RelatedElements", []) or []:
            if el.is_a("IfcWall") or el.is_a("IfcWallStandardCase"):
                wall_to_spaces[el.GlobalId].append(space)

    results = []
    for w in walls:
        spaces = wall_to_spaces.get(w.GlobalId, [])
        mapped = []
        for s in spaces:
            s_guid = getattr(s, "GlobalId", None)
            s_name = getattr(s, "Name", None)
            s_long = getattr(s, "LongName", None)

            # Strategy 1: GUID match (preferred)
            match      = by_guid.get(s_guid)
            match_type = "guid" if match else None

            # Strategy 2: name/number match (fallback)
            if not match:
                candidates = by_number.get(norm(s_name), []) + by_number.get(norm(s_long), [])
                match      = candidates[0] if candidates else None
                match_type = "space_number" if candidates else None

            mapped.append({
                "ifc_space_global_id":  s_guid,
                "ifc_space_name":       s_name,
                "ifc_space_long_name":  s_long,
                "mapping":              match,       # Full space_mapping entry, or null if unmatched
                "mapping_match_type":   match_type   # "guid", "space_number", or null
            })
        results.append({
            "wall_global_id": w.GlobalId,
            "wall_name":      getattr(w, "Name", None),
            "ifc_type":       w.is_a(),
            "linked_spaces":  mapped
        })

    os.makedirs(os.path.dirname(out_path) or "data", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} walls -> {out_path}")
    return results


out = link_walls_to_space_mapping()
print("Sample (first 3):")
print(json.dumps(out[:3], indent=2, ensure_ascii=False))

Saved 2649 walls -> data/walls_with_space_mapping.json
Sample (first 3):
[
  {
    "wall_global_id": "3w9amd8qrEXwHHbfc_$IZl",
    "wall_name": "Basic Wall:US1 (Ulkoseinä, teräsranka):481117",
    "ifc_type": "IfcWall",
    "linked_spaces": [
      {
        "ifc_space_global_id": "0eoqdncsb8SAoZq$ETTO6Y",
        "ifc_space_name": "1.kerros",
        "ifc_space_long_name": "1.kerros",
        "mapping": null,
        "mapping_match_type": null
      }
    ]
  }
]


---
## Step 5 — Gather all elements per space → `data/space_related_elements.json`

**Purpose:** For each space, collect every building element related to it. This data is used by Blender to highlight all geometry belonging to a space when a sensor state changes.

Four relationship types are queried (in priority order):
1. `IfcRelSpaceBoundary` — boundary elements (walls, slabs, doors)
2. `IfcRelContainedInSpatialStructure` — elements directly inside the space
3. `IfcRelCoversSpaces` — ceiling / floor coverings
4. Container fallback — any `IfcProduct` whose container is the space

> **Note:** Step 4 (fallback via container) iterates over all products and is intentionally slow for completeness. If performance is an issue, comment out that block.

> **Prerequisite:** Run Step 2 first to produce `data/space_mapping.json`.

In [4]:
# Step 5: For each IfcSpace, gather all related building elements.
# Produces a rich per-space dataset used by Blender for geometry highlighting.

import ifcopenshell
import ifcopenshell.util.element as el_util
import json
import os
from collections import defaultdict

# ⚠️ Change this filename if your IFC model has a different name
IFC_FILE = 'ARK_MET_F1.ifc'


def gather_space_relations(
    ifc_path=IFC_FILE,
    mapping_path="data/space_mapping.json",
    out_path="data/space_related_elements.json"
):
    model  = ifcopenshell.open(ifc_path)
    spaces = model.by_type("IfcSpace")

    # Load space mapping for cross-referencing (optional but useful)
    space_map = {}
    if os.path.exists(mapping_path):
        with open(mapping_path, "r", encoding="utf-8") as f:
            for entry in json.load(f):
                if entry.get("space_guid"):
                    space_map[entry["space_guid"]] = entry

    def add_element(target_list, el):
        """Add an element to the list, deduplicating by GlobalId."""
        if el is None:
            return
        gid = getattr(el, "GlobalId", None)
        if not gid or gid in seen:
            return
        seen.add(gid)
        psets = {}
        try:
            psets = el_util.get_psets(el) or {}
        except Exception:
            psets = {}
        target_list.append({
            "global_id":   gid,
            "ifc_type":    el.is_a(),
            "name":        getattr(el, "Name", None),
            "object_type": getattr(el, "ObjectType", None),
            "psets":       psets
        })

    results = []
    for s in spaces:
        seen          = set()
        related_items = []

        # 1) Explicit boundary elements (walls, slabs, doors, windows)
        for rel in model.by_type("IfcRelSpaceBoundary"):
            if getattr(rel, "RelatingSpace", None) is s:
                related = getattr(rel, "RelatedBuildingElement", None)
                if isinstance(related, (list, tuple)):
                    for r in related:
                        add_element(related_items, r)
                else:
                    add_element(related_items, related)

        # 2) Elements directly contained in the space
        for rel in model.by_type("IfcRelContainedInSpatialStructure"):
            if getattr(rel, "RelatingStructure", None) is s:
                for el in getattr(rel, "RelatedElements", []) or []:
                    add_element(related_items, el)

        # 3) Ceiling / floor coverings assigned to the space
        for rel in model.by_type("IfcRelCoversSpaces"):
            if getattr(rel, "RelatedSpace", None) is s:
                for el in getattr(rel, "RelatedCoverings", []) or []:
                    add_element(related_items, el)

        # 4) Fallback: any product whose direct container is this space.
        # Note: this pass iterates all IfcProducts and is the slowest step.
        # Comment it out if performance is critical and steps 1-3 are sufficient.
        for candidate in model.by_type("IfcProduct"):
            try:
                container = el_util.get_container(candidate)
            except Exception:
                container = None
            if container is s:
                add_element(related_items, candidate)

        mapping = space_map.get(getattr(s, "GlobalId", None))
        results.append({
            "space_global_id":   getattr(s, "GlobalId", None),
            "space_name":        getattr(s, "Name", None),
            "space_long_name":   getattr(s, "LongName", None),
            "space_mapping":     mapping,          # Cross-reference to space_mapping.json
            "related_elements":  related_items
        })

    os.makedirs(os.path.dirname(out_path) or "data", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(results)} spaces -> {out_path}")
    return results


out = gather_space_relations()
print("Sample (first 2 spaces):")
print(json.dumps(out[:2], indent=2, ensure_ascii=False))

Saved 222 spaces -> data/space_related_elements.json
Sample (first 2 spaces):


## Extra - Reconstruct IfcSpace for web-ifc and Three.js

---
## Step 6a — Strip IfcPresentationLayerAssignment → visible in Blender & web-ifc

**Why:** Revit exports place `IfcSpace` on a presentation layer with `LayerOn = False`.
Both Bonsai and web-ifc respect this flag, so spaces are invisible by default.

Two options are offered:
- **Option A** — set `LayerOn = True` on every layer (safer, keeps layer metadata).
- **Option B** — remove all `IfcPresentationLayerAssignment` entities entirely
  (simpler, no layer data left in the file).

Run this **before** Step 6b (numeric space removal) on your source IFC.
Output: `ARK_MET_F1_layers_on.ifc`


In [5]:
# Step 6a: Fix IfcPresentationLayerAssignment so IfcSpaces are visible
# in Blender/Bonsai and web-ifc without changing geometry or metadata.

import ifcopenshell

# ⚠️ Change paths as needed
IFC_INPUT  = "ARK_MET_F1.ifc"
IFC_OUTPUT = "ARK_MET_F1_layers_on.ifc"

# True = Option A (turn layers on)  |  False = Option B (remove them)
KEEP_LAYERS = True


def fix_layer_visibility(input_path, output_path, keep_layers=True):
    model  = ifcopenshell.open(input_path)
    layers = model.by_type("IfcPresentationLayerAssignment")
    print(f"  IfcPresentationLayerAssignment found: {len(layers)}")

    if not layers:
        print("  No layer assignments found — nothing to fix.")
        model.write(output_path)
        return

    if keep_layers:
        # Option A: flip every explicitly-off layer to on
        # LayerOn is IfcLogical: True / False / UNKNOWN (None)
        changed = 0
        for layer in layers:
            if getattr(layer, 'LayerOn', None) is False:
                layer.LayerOn = True
                changed += 1
        print(f"  Option A: turned ON {changed} previously-off layers.")
    else:
        # Option B: remove all layer assignment entities entirely
        for layer in layers:
            model.remove(layer)
        print(f"  Option B: removed all {len(layers)} layer assignments.")

    model.write(output_path)
    print(f"  ✅ Saved: {output_path}")


fix_layer_visibility(IFC_INPUT, IFC_OUTPUT, KEEP_LAYERS)

# Use ARK_MET_F1_layers_on.ifc as the input for Step 6b (numeric space removal).


  IfcPresentationLayerAssignment found: 0
  No layer assignments found — nothing to fix.


---
## Step 6b — Remove numeric-only IfcSpaces → `ARK_MET_F1_clean.ifc`

**Purpose:** Produce a clean IFC model containing only named rooms (spaces whose `Name`
starts with a capital letter such as A, B, C, D). Purely numeric spaces (e.g. `"1"`, `"2"`,
`"899"`) are zone/area placeholders with no sensor relevance.

**Input:** Use `ARK_MET_F1_layers_on.ifc` from Step 6a — not the original file.

**What is removed:** Every `IfcSpace` whose `Name` is digits only, plus all relationship
objects referencing those spaces so the IFC graph stays consistent.

**What is preserved:** All named spaces and their `GlobalId`, `Name`, `LongName`, geometry,
and property sets — the fields read by `space_mapping.json` are untouched.

> **Prerequisite:** Run Step 2 first so `data/space_mapping.json` exists for the
> validation check at the end.


In [6]:
# Step 6b: Remove numeric-only IfcSpaces and save a clean IFC model.
# Run AFTER Step 6a so layer visibility is already fixed in the input file.

import ifcopenshell
import json
import os

# ⚠️ Use the layers-fixed file from Step 6a as input
IFC_INPUT          = "ARK_MET_F1_layers_on.ifc"
IFC_OUTPUT         = "ARK_MET_F1_clean.ifc"
SPACE_MAPPING_JSON = "data/space_mapping.json"


def is_numeric_only(name):
    return str(name or "").strip().isdigit()


def collect_referencing_rels(model, space_ids):
    to_remove = set()
    for rel in model.by_type("IfcRelAggregates"):
        if any(o.id() in space_ids for o in (getattr(rel, "RelatedObjects", None) or [])):
            to_remove.add(rel.id())
    for rel in model.by_type("IfcRelContainedInSpatialStructure"):
        if any(o.id() in space_ids for o in (getattr(rel, "RelatedElements", None) or [])):
            to_remove.add(rel.id())
    for rel in model.by_type("IfcRelSpaceBoundary"):
        space = getattr(rel, "RelatingSpace", None)
        if space is not None and space.id() in space_ids:
            to_remove.add(rel.id())
    for rel in model.by_type("IfcRelDefinesByProperties"):
        if any(o.id() in space_ids for o in (getattr(rel, "RelatedObjects", None) or [])):
            to_remove.add(rel.id())
    for rel in model.by_type("IfcRelAssociatesConstraint"):
        if any(o.id() in space_ids for o in (getattr(rel, "RelatedObjects", None) or [])):
            to_remove.add(rel.id())
    return to_remove


def validate_against_space_mapping(model, mapping_path):
    if not os.path.exists(mapping_path):
        print(f"  [skip validation] {mapping_path} not found — run Step 2 first.")
        return
    with open(mapping_path, encoding="utf-8") as f:
        mapping = json.load(f)
    kept_entries   = [e for e in mapping if not is_numeric_only(e.get("space_number"))]
    existing_guids = {s.GlobalId for s in model.by_type("IfcSpace")}
    missing        = [e for e in kept_entries if e["space_guid"] not in existing_guids]
    if missing:
        print(f"\n  ⚠️  {len(missing)} space_mapping entries NOT found in cleaned model:")
        for e in missing:
            print(f"       {e['space_number']:<12}  {e['space_guid']}")
    else:
        print(f"\n  ✅  All {len(kept_entries)} non-numeric space_mapping entries "
              f"are present — sensor pipeline unaffected.")


def remove_numeric_spaces(input_path, output_path, mapping_path=SPACE_MAPPING_JSON):
    print("=" * 60)
    print("  Step 6b — Remove numeric-only IfcSpaces")
    print("=" * 60)
    if not os.path.exists(input_path):
        print(f"  ✗  Input not found: {input_path}")
        return
    model      = ifcopenshell.open(input_path)
    all_spaces = model.by_type("IfcSpace")
    print(f"\n  IfcSpace total (before)  : {len(all_spaces)}")
    numeric    = [s for s in all_spaces if is_numeric_only(s.Name)]
    num_ids    = {s.id() for s in numeric}
    print(f"  Numeric-only (to remove) : {len(numeric)}")
    print(f"  Named rooms  (to keep)   : {len(all_spaces) - len(numeric)}")
    if not numeric:
        print("\n  Nothing to remove."); return
    print("\n  Spaces being removed:")
    for s in sorted(numeric, key=lambda x: str(x.Name)):
        print(f"    Name={str(s.Name):<6}  LongName={str(getattr(s,'LongName',None) or ''):<28}  GUID={s.GlobalId}")
    rel_ids = collect_referencing_rels(model, num_ids)
    print(f"\n  Referencing relationships to remove: {len(rel_ids)}")
    removed_rels = sum(1 for rid in rel_ids if not __import__('builtins').setattr(model, '_', model.remove(model.by_id(rid))) or True)
    # simpler loop:
    removed_rels = 0
    for rid in rel_ids:
        try: model.remove(model.by_id(rid)); removed_rels += 1
        except Exception as e: print(f"  [warn] rel #{rid}: {e}")
    removed_spaces = 0
    for s in numeric:
        try: model.remove(s); removed_spaces += 1
        except Exception as e: print(f"  [warn] space {s.Name}: {e}")
    print(f"\n  Removed spaces            : {removed_spaces}")
    print(f"  Removed relationships     : {removed_rels}")
    print(f"  IfcSpace total (after)    : {len(model.by_type('IfcSpace'))}")
    validate_against_space_mapping(model, mapping_path)
    model.write(output_path)
    print(f"\n  ✅  Saved: {output_path}")
    print("=" * 60 + "\n")


remove_numeric_spaces(IFC_INPUT, IFC_OUTPUT, SPACE_MAPPING_JSON)


  Step 6b — Remove numeric-only IfcSpaces

  IfcSpace total (before)  : 222
  Numeric-only (to remove) : 22
  Named rooms  (to keep)   : 200

  Spaces being removed:
    Name=1       LongName=Huoneistoala 1-kerros         GUID=3dqv97I1b2JR60ciMz8rBL
    Name=1       LongName=KOKOONTUMISTILA 1             GUID=02ZODw3PX4WPbIVq0FTGUh
    Name=1       LongName=LIIKETILA                     GUID=21VEzGVNLChPldQbstk9d4
    Name=11      LongName=KERROSALA 1-kerros            GUID=27fqKLTvPCgvbgPSh1T1ty
    Name=128     LongName=1                             GUID=0pOcnfSJLEyAfGEqQAWHJE
    Name=132     LongName=1                             GUID=0I09zXfwn4HP3vNed_nmFo
    Name=133     LongName=1                             GUID=0I09zXfwn4HP3vNed_nmFz
    Name=2       LongName=KERROSALA 1-kerros            GUID=0QiU8cOrnAZ8CrW1VZg$UA
    Name=2       LongName=KOKOONTUMISTILA 2             GUID=02ZODw3PX4WPbIVq0FTGL3
    Name=2       LongName=MAHDOLLINEN LIIKETILA         GUID=21VEzGVNLChPldQbs

---
## ✅ Done — Next steps

After all cells run successfully, the `data/` folder contains:

```
data/
├── space_mapping.json              ← used by sensor mapping scripts
├── walls_with_spaces.json          ← intermediate (can be deleted after Step 4)
├── walls_with_space_mapping.json   ← used by Blender wall colouring
└── space_related_elements.json     ← used by Blender element highlighting
```

You are now ready to proceed to **fetching sensor data** from the Empathic Building API.
See `docs/SETUP_AND_RUNNING.md` for the full workflow.